# Graduate Admission Chance Prediction Using Machine Learning

## YBI Foundation — 6-Month Live Project-Based Internship

### Project Type
Machine Learning Regression Project

### Objective
The objective of this project is to build a machine learning model that predicts a student's **Chance of Admission** to a graduate program using academic profile information.

### Input Features
- GRE Score
- TOEFL Score
- University Rating
- Statement of Purpose (SOP)
- Letter of Recommendation (LOR)
- CGPA
- Research Experience

### Target Variable
- Chance of Admit

## 1. Import Libraries

The required Python libraries are imported for data handling, visualization, model training, and evaluation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load the Dataset

The dataset is loaded directly from the YBI Foundation dataset repository, so no manual file upload is required in Google Colab.

In [ ]:
DATA_URL = "https://github.com/ybifoundation/Dataset/raw/main/Admission%20Chance.csv"

admission = pd.read_csv(DATA_URL)
print("Dataset loaded successfully.")
print("Dataset shape:", admission.shape)

admission.head()

## 3. Understand the Dataset

In [ ]:
print("Column names:")
print(admission.columns.tolist())

print("\nDataset information:")
admission.info()

In [ ]:
print("Statistical summary:")
admission.describe().T

## 4. Data Cleaning

Column names are cleaned to remove extra spaces. The serial number is only an identifier and does not help predict admission chances, so it is removed.

In [ ]:
admission.columns = admission.columns.str.strip()

if "Serial No" in admission.columns:
    admission = admission.drop(columns=["Serial No"])

print("Cleaned columns:")
print(admission.columns.tolist())

print("\nMissing values:")
print(admission.isnull().sum())

print("\nDuplicate rows:", admission.duplicated().sum())

In [ ]:
# Remove duplicate rows, if any
admission = admission.drop_duplicates().reset_index(drop=True)

print("Final dataset shape after cleaning:", admission.shape)
admission.head()

## 5. Exploratory Data Analysis

In [ ]:
# Distribution of the target variable
plt.figure(figsize=(8, 5))
plt.hist(admission["Chance of Admit"], bins=15, edgecolor="black")
plt.title("Distribution of Chance of Admit")
plt.xlabel("Chance of Admit")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
# Correlation matrix
correlation = admission.corr(numeric_only=True)

plt.figure(figsize=(10, 7))
plt.imshow(correlation, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=45, ha="right")
plt.yticks(range(len(correlation.columns)), correlation.columns)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

print("Correlation with Chance of Admit:")
print(correlation["Chance of Admit"].sort_values(ascending=False))

### EDA Observation

The correlation analysis helps identify which academic factors have the strongest relationship with graduate admission chances. In general, stronger academic scores and profile quality are expected to be associated with a higher chance of admission.

## 6. Define Features (X) and Target (y)

In [ ]:
X = admission.drop(columns=["Chance of Admit"])
y = admission["Chance of Admit"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("\nFeatures used:")
print(X.columns.tolist())

## 7. Split the Data

The dataset is divided into training and testing sets. The model learns from 80% of the data and is evaluated on the remaining 20%.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 8. Train Machine Learning Models

Two regression models are trained and compared:

1. Linear Regression
2. Random Forest Regressor

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

random_forest_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)
random_forest_model.fit(X_train, y_train)

print("Both models trained successfully.")

## 9. Model Evaluation

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    }, predictions

linear_result, linear_predictions = evaluate_model(
    "Linear Regression", linear_model, X_test, y_test
)

rf_result, rf_predictions = evaluate_model(
    "Random Forest Regressor", random_forest_model, X_test, y_test
)

results = pd.DataFrame([linear_result, rf_result])
results

In [ ]:
best_model_name = results.loc[results["R2 Score"].idxmax(), "Model"]
print("Best model based on R2 Score:", best_model_name)

### Evaluation Metrics

- **MAE (Mean Absolute Error):** Average absolute difference between actual and predicted values. Lower is better.
- **RMSE (Root Mean Squared Error):** Gives more penalty to larger prediction errors. Lower is better.
- **R² Score:** Shows how well the model explains variation in the target. Closer to 1 is better.

## 10. Actual vs Predicted Values

In [ ]:
best_predictions = (
    linear_predictions
    if best_model_name == "Linear Regression"
    else rf_predictions
)

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": best_predictions
})

comparison["Absolute Error"] = abs(comparison["Actual"] - comparison["Predicted"])
comparison.head(10)

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.7)

minimum = min(y_test.min(), best_predictions.min())
maximum = max(y_test.max(), best_predictions.max())
plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")

plt.xlabel("Actual Chance of Admit")
plt.ylabel("Predicted Chance of Admit")
plt.title(f"Actual vs Predicted — {best_model_name}")
plt.show()

## 11. Feature Importance / Model Interpretation

In [ ]:
if best_model_name == "Random Forest Regressor":
    importance = pd.Series(
        random_forest_model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)
else:
    importance = pd.Series(
        np.abs(linear_model.coef_),
        index=X.columns
    ).sort_values(ascending=False)

importance_df = importance.reset_index()
importance_df.columns = ["Feature", "Importance"]
importance_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(importance_df["Feature"][::-1], importance_df["Importance"][::-1])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title(f"Feature Importance — {best_model_name}")
plt.tight_layout()
plt.show()

## 12. Predict Admission Chance for a New Student

The following example demonstrates how the trained model can predict the admission chance for a new student profile.

In [ ]:
best_model = (
    linear_model
    if best_model_name == "Linear Regression"
    else random_forest_model
)

new_student = pd.DataFrame([{
    "GRE Score": 320,
    "TOEFL Score": 110,
    "University Rating": 4,
    "SOP": 4.0,
    "LOR": 4.0,
    "CGPA": 8.80,
    "Research": 1
}])

# Ensure the exact same feature order used during training
new_student = new_student[X.columns]

predicted_chance = best_model.predict(new_student)[0]
predicted_chance = np.clip(predicted_chance, 0, 1)

print(f"Predicted Chance of Admission: {predicted_chance:.2%}")

## 13. Conclusion

This project successfully developed a machine learning solution to predict the chance of graduate admission from a student's academic profile.

### Work Completed
- Loaded and explored the admission dataset
- Cleaned column names and checked data quality
- Performed exploratory data analysis
- Split the dataset into training and testing sets
- Trained Linear Regression and Random Forest Regression models
- Compared the models using MAE, RMSE, and R² Score
- Selected the better-performing model
- Visualized actual and predicted values
- Interpreted feature importance
- Predicted the admission chance for a new student

### Final Result
The model comparison identifies the best-performing regression algorithm for this dataset. The completed pipeline demonstrates a practical application of machine learning in educational decision support.

### Future Scope
The project can be extended by:
- Testing additional regression algorithms
- Performing hyperparameter tuning
- Deploying the model as a web application
- Adding a user interface for real-time predictions

## Final Project Summary

**Project Title:** Graduate Admission Chance Prediction Using Machine Learning

This project demonstrates a complete machine learning regression workflow, including data loading, data cleaning, exploratory data analysis, model training, model comparison, evaluation, interpretation, and prediction for a new student profile.

The project was completed as part of the **YBI Foundation 6-Month Live Project-Based Internship**.
